# mixed-mTRF: Bayesian Temporal Response Functions with GP Prior
### Synthetic data validation notebook

**Author:** Ella Capellini  
**Lab:** Neural Dynamics Lab, INI — University of Zurich  
**Supervisor:** Professor Dr. Timothee Piox  

---

## What this notebook does

This notebook validates the full `mixed-mTRF` pipeline on synthetic data
where the ground truth TRF is known. The logic is:

1. Simulate a realistic speech-like stimulus + EEG signal with a known TRF
2. Fit two models: the standard **ridge mTRF** (baseline) and the new
   **GP-mTRF** (Bayesian, with Matérn 5/2 prior)
3. Compare both against the ground truth
4. Prove that ridge regression is a mathematical special case of GP-mTRF
5. Show what the GP gives you that ridge cannot: **uncertainty**

Once this passes, the only thing that changes for real iEEG data is
the data loading step. The models are identical.

## Background: what is a TRF?

A **Temporal Response Function** is the brain's impulse response to a
continuous stimulus feature. It answers: *when the stimulus does X,
what does the brain do over the next ~800ms?*

The fundamental relationship:

$$r(t) \approx \sum_\tau h(\tau) \cdot s(t - \tau)$$

- $s(t)$ = stimulus feature at time $t$ (e.g. speech envelope, surprisal, lip aperture)
- $h(\tau)$ = TRF weight at lag $\tau$ — **this is what we estimate**
- $r(t)$ = EEG signal at time $t$

To estimate $h$, we build a **design matrix** $X$ where each column is
the feature shifted by a different lag, then solve a regression problem:

$$\hat{h} = \arg\min_h \|Xh - r\|^2 + \text{regularisation}$$

**Ridge regression** adds $\lambda \|h\|^2$ as the regularisation term —
this is the standard mTRF. It gives one fixed estimate with no uncertainty.

**GP-mTRF** replaces the ridge penalty with a Gaussian Process prior
that encodes *smoothness*: nearby lags should have correlated TRF values.
This gives a full posterior distribution over TRF shapes — mean + uncertainty.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize

# Reproducibility
rng = np.random.default_rng(42)

# Plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.color':       '#EEEEEE',
    'grid.linewidth':   0.8,
    'font.size':        10,
})

COLORS = {
    'truth': '#2D2D2D',
    'ridge': '#E07B39',
    'gp':    '#4A7FB5',
    'ci':    '#4A7FB5',
}

print("Imports OK")

ModuleNotFoundError: No module named 'numpy'

## Step 1: Build the design matrix

The design matrix $X$ has shape `(n_times, n_lags)`.
Column $i$ = the stimulus shifted by `lags[i]` samples.

Row $t$, column $i$ answers: *"what was the stimulus `lags[i]` samples
ago, at timepoint $t$?"*

When we multiply $X \cdot h$, we reconstruct the predicted EEG as a
weighted sum of past stimulus values — exactly the convolution model above.

For **multiple features** (e.g. envelope + surprisal + lip aperture),
we stack design matrices horizontally:

$$X = [\underbrace{X_\text{env}}_{n_\text{lags}} \mid 
       \underbrace{X_\text{surp}}_{n_\text{lags}} \mid 
       \underbrace{X_\text{lip}}_{n_\text{lags}}]$$

Each TRF is then estimated *controlling for all other features simultaneously*.
This is what makes it **multivariate**.

def build_design_matrix(stimulus, lags, fill_value=0.0):
    """
    Build a lagged design matrix from a single stimulus feature.
    
    Parameters
    ----------
    stimulus : np.ndarray, shape (n_times,)
    lags     : np.ndarray of int — lag grid IN SAMPLES (not ms)
               convert: lags = np.arange(tmin_ms * sr // 1000,
                                          tmax_ms * sr // 1000 + 1)
    
    Returns
    -------
    X : np.ndarray, shape (n_times, n_lags)
        X[:, i] = stimulus shifted by lags[i] samples
    """
    n_times = len(stimulus)
    X = np.full((n_times, len(lags)), fill_value)
    for i, lag in enumerate(lags):
        if lag >= 0:
            # Positive lag: brain responds AFTER stimulus
            X[lag:, i] = stimulus[:n_times - lag]
        else:
            # Negative lag: pre-stimulus window
            # (should be ~0 if there is no anticipatory response)
            X[:n_times + lag, i] = stimulus[-lag:]
    return X


def build_multifeature_design_matrix(features, lags):
    """
    Stack design matrices for multiple features horizontally.
    
    Parameters
    ----------
    features : dict of {str: np.ndarray, shape (n_times,)}
    lags     : shared lag grid in samples
    
    Returns
    -------
    X              : np.ndarray, shape (n_times, n_features * n_lags)
    feature_slices : dict mapping feature name → column slice in X
                     use to extract individual TRFs after fitting
    """
    n_lags = len(lags)
    blocks, feature_slices, col = [], {}, 0
    for name, feat in features.items():
        blocks.append(build_design_matrix(feat, lags))
        feature_slices[name] = slice(col, col + n_lags)
        col += n_lags
    return np.hstack(blocks), feature_slices


# Quick sanity check
sr   = 100   # Hz — standard for envelope-level TRF analysis
lags = np.arange(int(-0.1 * sr), int(0.8 * sr) + 1)  # -100ms to +800ms
lag_times = lags / sr  # in seconds, for plotting

test_stim = np.zeros(200)
test_stim[50] = 1.0  # single impulse at t=50
X_test = build_design_matrix(test_stim, lags)

lag0_col = np.where(lags == 0)[0][0]
assert X_test[50, lag0_col] == 1.0,   "Lag 0 should align with impulse"
assert X_test[60, lag0_col+10] == 1.0,"Lag 10 samples should shift correctly"

print(f"Lag grid: {lags[0]} to {lags[-1]} samples "
      f"= {lags[0]/sr*1000:.0f}ms to {lags[-1]/sr*1000:.0f}ms")
print(f"Number of lags: {len(lags)}")
print("Design matrix: ✓")

## Step 2: Kernels — encoding prior beliefs about the TRF

A **kernel** $k(\tau, \tau')$ defines the covariance between TRF values
at two lags. It encodes what we believe about the TRF *before seeing data*.

**Key intuition:** as $|\tau - \tau'|$ grows, $k(\tau, \tau') \to 0$
(distant lags are uncorrelated). When $\tau = \tau'$, $k = \sigma^2$
(maximum self-correlation). This forces the TRF to be smooth —
it literally cannot jump discontinuously between adjacent lags.

We implement two kernels:

| Kernel | Formula | Properties |
|--------|---------|-----------|
| Squared Exponential | $\sigma^2 \exp(-\frac{(\tau-\tau')^2}{2\ell^2})$ | Infinitely smooth |
| Matérn 5/2 | $\sigma^2(1+\frac{\sqrt5 r}{\ell}+\frac{5r^2}{3\ell^2})e^{-\frac{\sqrt5 r}{\ell}}$ | Twice differentiable, sharper peaks |

We use **Matérn 5/2** as default — it allows slightly sharper features
(like the N1 peak at ~100ms) that SE would over-smooth.

And **WhiteNoise** kernel: $k(\tau, \tau') = \sigma^2 \delta(\tau, \tau')$
— all lags independent. We will prove this recovers ridge regression exactly.

In [ ]:
class SquaredExponential:
    """
    SE kernel: k(τ,τ') = σ² · exp(−(τ−τ')²/2ℓ²)
    Infinitely smooth — good default, slightly over-smooths sharp peaks.
    
    Parameters
    ----------
    output_scale : σ² — controls overall TRF amplitude scale
    length_scale : ℓ  — controls smoothness (in samples)
                        rule of thumb: ~width of narrowest ERP component
    """
    def __init__(self, output_scale=1.0, length_scale=5.0):
        self.output_scale = output_scale
        self.length_scale = length_scale
    def __call__(self, t1, t2):
        d = t1[:, None] - t2[None, :]
        return self.output_scale * np.exp(-d**2 / (2 * self.length_scale**2))
    def copy(self):
        return SquaredExponential(self.output_scale, self.length_scale)


class Matern52:
    """
    Matérn 5/2 kernel — default for neural data.
    Slightly rougher than SE: twice differentiable (not infinitely smooth).
    Better for sharp ERP peaks like the N1.
    """
    def __init__(self, output_scale=1.0, length_scale=5.0):
        self.output_scale = output_scale
        self.length_scale = length_scale
    def __call__(self, t1, t2):
        d = t1[:, None] - t2[None, :]
        r = np.abs(d) / self.length_scale
        s = np.sqrt(5) * r
        return self.output_scale * (1 + s + (5/3)*r**2) * np.exp(-s)
    def copy(self):
        return Matern52(self.output_scale, self.length_scale)


class WhiteNoise:
    """
    White noise kernel: k(τ,τ') = σ² · δ(τ,τ')
    Completely diagonal — every lag independent.
    This is the IMPLICIT prior in ridge regression.
    Setting output_scale = noise_var / alpha recovers ridge exactly.
    """
    def __init__(self, output_scale=1.0):
        self.output_scale = output_scale
    def __call__(self, t1, t2):
        return self.output_scale * (t1[:, None] == t2[None, :]).astype(float)


# Visualise the kernel matrix
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
lag_ms = lag_times * 1000

for ax, (name, kernel) in zip(axes, [
    ('Squared Exponential\n(ℓ=50ms)', SquaredExponential(1.0, 5.0)),
    ('Matérn 5/2\n(ℓ=50ms, default)', Matern52(1.0, 5.0)),
    ('White Noise\n(= ridge prior)', WhiteNoise(1.0)),
]):
    K = kernel(lags, lags)
    im = ax.imshow(K, aspect='auto', cmap='Blues',
                   extent=[lag_ms[0], lag_ms[-1], lag_ms[-1], lag_ms[0]])
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Lag (ms)')
    ax.set_ylabel('Lag (ms)')

plt.suptitle('Prior covariance matrices K — what each kernel believes about TRF smoothness',
             y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

# The diagonal band = nearby lags are correlated (smooth TRF)
# White noise = diagonal only = ridge = no smoothness assumption
print("Kernel visualisation: ✓")
print("Notice: SE and Matérn have off-diagonal structure (smoothness)")
print("White noise is diagonal only — this is what ridge implicitly assumes")

## Step 3: Simulate synthetic data

We simulate:
- A **sparse stimulus** at ~2 events/second with random amplitudes
  (mimics word onsets with varying surprisal)
- A **known ground truth TRF** with three ERP components:
  - **N1** at 100ms (early auditory response, negative)
  - **P2** at 200ms (positive)
  - **N400** at 400ms (semantic/predictability, negative)
- **Two features**: envelope (strong) + surprisal (weaker, shifted)
- **Gaussian noise** at SNR=2 (realistic for iEEG)

The EEG is: $r(t) = \sum_\tau h(\tau) \cdot s(t-\tau) + \epsilon(t)$

Since we *know* the true TRF, we can directly measure whether
each model recovers it correctly.

In [ ]:
def make_erp_trf(lag_times, shift=0.0, scale=1.0):
    """
    Realistic ERP-like ground truth TRF.
    Three Gaussian components: N1, P2, N400.
    shift : seconds — time-shifts all components
    scale : overall amplitude scaling
    """
    t = lag_times - shift
    trf  = -1.5 * np.exp(-0.5 * ((t - 0.10) / 0.04)**2)  # N1  at 100ms
    trf +=  1.0 * np.exp(-0.5 * ((t - 0.20) / 0.05)**2)  # P2  at 200ms
    trf += -0.5 * np.exp(-0.5 * ((t - 0.40) / 0.10)**2)  # N400 at 400ms
    return scale * trf


# Simulation parameters
n_times = 12000   # 120 seconds at 100Hz

# Two ground truth TRFs — different amplitude and timing
true_trf_env  = make_erp_trf(lag_times, shift=0.00, scale=1.0)
true_trf_surp = make_erp_trf(lag_times, shift=0.05, scale=0.5)

# Two sparse stimulus features
stim_env  = np.zeros(n_times)
stim_surp = np.zeros(n_times)
ev_env  = rng.choice(n_times, size=int(2 * n_times / sr), replace=False)
ev_surp = rng.choice(n_times, size=int(1.5 * n_times / sr), replace=False)
stim_env[ev_env]   = rng.exponential(1.0, size=len(ev_env))
stim_surp[ev_surp] = rng.exponential(0.8, size=len(ev_surp))

# Build design matrix
X, slices = build_multifeature_design_matrix(
    {'envelope': stim_env, 'surprisal': stim_surp}, lags
)

# EEG = convolution + noise
eeg_clean = (X[:, slices['envelope']]  @ true_trf_env +
             X[:, slices['surprisal']] @ true_trf_surp)
true_noise_var = (np.std(eeg_clean) / 2)**2   # SNR = 2
eeg = eeg_clean + rng.normal(0, np.sqrt(true_noise_var), n_times)
snr = np.std(eeg_clean) / np.sqrt(true_noise_var)

print(f"n_times         : {n_times} samples = {n_times/sr:.0f} seconds")
print(f"Design matrix   : {X.shape}")
print(f"True noise_var  : {true_noise_var:.4f}")
print(f"SNR             : {snr:.2f}")
print(f"envelope events : {(stim_env > 0).sum()}")
print(f"surprisal events: {(stim_surp > 0).sum()}")

# Plot ground truth TRFs and a snippet of EEG
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
lag_ms = lag_times * 1000

ax = axes[0]
ax.axhline(0, color='#CCCCCC', lw=0.8)
ax.axvline(0, color='#CCCCCC', lw=0.8, ls='--')
ax.plot(lag_ms, true_trf_env,  color=COLORS['truth'], lw=2,
        label='envelope TRF (scale=1.0)')
ax.plot(lag_ms, true_trf_surp, color=COLORS['truth'], lw=2, ls='--', alpha=0.6,
        label='surprisal TRF (scale=0.5, +50ms shift)')
ax.annotate('N1',   xy=(100, -1.5), xytext=(130, -1.65),
            arrowprops=dict(arrowstyle='->', color='#999', lw=0.8), fontsize=9)
ax.annotate('P2',   xy=(200,  1.0), xytext=(230,  1.1),
            arrowprops=dict(arrowstyle='->', color='#999', lw=0.8), fontsize=9)
ax.annotate('N400', xy=(400, -0.5), xytext=(430, -0.65),
            arrowprops=dict(arrowstyle='->', color='#999', lw=0.8), fontsize=9)
ax.legend(fontsize=8, frameon=False)
ax.set_xlabel('Lag (ms)')
ax.set_ylabel('Amplitude (a.u.)')
ax.set_title('Ground truth TRFs')

ax = axes[1]
t_snip = np.arange(500) / sr
ax.plot(t_snip, eeg[:500], color='#AAAAAA', lw=0.7, label='EEG (with noise)')
ax.plot(t_snip, eeg_clean[:500], color=COLORS['truth'], lw=1.5,
        alpha=0.8, label='clean signal')
ax.legend(fontsize=8, frameon=False)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Voltage (a.u.)')
ax.set_title(f'Simulated EEG — 5s snippet (SNR={snr:.1f})')

plt.tight_layout()
plt.show()
print("Synthetic data: ✓")

## Step 4: Ridge mTRF — the baseline

Ridge regression solves:

$$\hat{h} = (X^\top X + \alpha I)^{-1} X^\top y$$

The $\alpha I$ term adds a small constant to the diagonal of $X^\top X$,
making it invertible and preventing overfitting.

**Cross-validation**: we use **contiguous time folds** — not random shuffling.
EEG is autocorrelated in time. Random shuffling would let information
"leak" across the train/test boundary and give over-optimistic scores.

The output is a single weight vector — one TRF value per lag per feature.
**No uncertainty information.**

In [ ]:
class RidgemTRF:
    """
    Standard multivariate TRF via ridge regression.
    Baseline model. Mathematical special case of GP-mTRF
    with a WhiteNoise kernel (proved below).
    """
    def __init__(self, lags, alpha=1.0):
        self.lags   = lags
        self.alpha  = alpha
        self.coef_  = None

    def fit(self, X, y):
        """ĥ = (XᵀX + αI)⁻¹ Xᵀy"""
        n = X.shape[1]
        # np.linalg.solve is more numerically stable than explicit inverse
        self.coef_ = np.linalg.solve(
            X.T @ X + self.alpha * np.eye(n), X.T @ y
        )
        return self

    def predict(self, X): return X @ self.coef_

    def score(self, X, y):
        """Pearson r — standard TRF evaluation metric"""
        return np.corrcoef(y, self.predict(X))[0, 1]

    def get_trf(self, name, slices):
        return self.coef_[slices[name]]

    def fit_cv(self, X, y, alphas, n_folds=5):
        """
        Cross-validate α using contiguous time folds.
        Refits on all data with best α.
        """
        n_times   = len(y)
        fold_size = n_times // n_folds
        cv_scores = np.zeros(len(alphas))

        for i, alpha in enumerate(alphas):
            corrs = []
            for fold in range(n_folds):
                mask = np.ones(n_times, dtype=bool)
                mask[fold*fold_size:(fold+1)*fold_size] = False
                tmp = RidgemTRF(self.lags, alpha)
                tmp.fit(X[mask], y[mask])
                corrs.append(tmp.score(X[~mask], y[~mask]))
            cv_scores[i] = np.mean(corrs)

        best_alpha = alphas[np.argmax(cv_scores)]
        self.alpha = best_alpha
        self.fit(X, y)
        return best_alpha, cv_scores


# Fit with cross-validation
alphas = np.logspace(-3, 4, 20)
ridge  = RidgemTRF(lags, alpha=1.0)
best_alpha, cv_scores = ridge.fit_cv(X, eeg, alphas, n_folds=5)

# Evaluate
ridge_trf_env  = ridge.get_trf('envelope',  slices)
ridge_trf_surp = ridge.get_trf('surprisal', slices)
r_env  = np.corrcoef(true_trf_env,  ridge_trf_env)[0, 1]
r_surp = np.corrcoef(true_trf_surp, ridge_trf_surp)[0, 1]

print(f"Best α (cross-validated): {best_alpha:.4f}")
print(f"Envelope  TRF recovery : r = {r_env:.4f}")
print(f"Surprisal TRF recovery : r = {r_surp:.4f}")

# CV curve
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].semilogx(alphas, cv_scores, color=COLORS['ridge'], lw=1.8, marker='o', ms=4)
axes[0].axvline(best_alpha, color='#888', lw=1, ls='--', label=f'best α={best_alpha:.3f}')
axes[0].legend(fontsize=8, frameon=False)
axes[0].set_xlabel('α (regularisation strength)')
axes[0].set_ylabel('Mean held-out Pearson r')
axes[0].set_title('Cross-validation curve')

axes[1].axhline(0, color='#CCCCCC', lw=0.8)
axes[1].axvline(0, color='#CCCCCC', lw=0.8, ls='--')
axes[1].plot(lag_ms, true_trf_env,  color=COLORS['truth'], lw=2.2, label='ground truth')
axes[1].plot(lag_ms, ridge_trf_env, color=COLORS['ridge'], lw=1.8, ls='--',
             label=f'ridge (r={r_env:.4f})')
axes[1].legend(fontsize=8, frameon=False)
axes[1].set_xlabel('Lag (ms)')
axes[1].set_ylabel('Amplitude (a.u.)')
axes[1].set_title('Ridge TRF — envelope feature')

plt.tight_layout()
plt.show()
print("Ridge mTRF: ✓")

## Step 5: GP-mTRF — the Bayesian model

The GP-mTRF replaces the ridge penalty with a GP prior over the TRF.

**Generative model:**

$$h \sim \mathcal{GP}(0, K) \quad \text{[prior — smooth TRF]}$$
$$y \mid h \sim \mathcal{N}(Xh, \sigma^2 I) \quad \text{[likelihood — noisy EEG]}$$

**Posterior** (exact, closed-form because both terms are Gaussian):

$$h \mid y \sim \mathcal{N}(\mu, \Sigma)$$

$$\Sigma = \left(K^{-1} + \frac{X^\top X}{\sigma^2}\right)^{-1}
\quad \mu = \frac{\Sigma X^\top y}{\sigma^2}$$

We work in **weight space** — inverting an $(n_\text{weights} \times n_\text{weights})$
matrix, not $(n_\text{times} \times n_\text{times})$. For 3 features × 91 lags,
that's a $273 \times 273$ inversion even for hours of data.

**What we gain over ridge:**
- $\mu$ = posterior mean TRF (same interpretation as ridge)
- $\sqrt{\text{diag}(\Sigma)}$ = **uncertainty at each lag** — ridge has none
- $\log p(y)$ = log marginal likelihood — used to learn hyperparameters from data

**Hyperparameters** (`output_scale`, `length_scale`, `noise_var`) are
optimised by maximising the log marginal likelihood via L-BFGS-B.
Each feature gets its own kernel — independent smoothness and amplitude priors.

In [ ]:
class GPmTRF:
    """
    Multivariate TRF with Gaussian Process prior.
    
    Parameters
    ----------
    lags     : lag grid in samples
    kernel   : SquaredExponential / Matern52 — OR list of kernels
               (one per feature, for independent priors per feature)
    noise_var: σ² — EEG observation noise variance
    """
    def __init__(self, lags, kernel, noise_var=1.0):
        self.lags      = lags
        self.noise_var = noise_var
        self.mean_ = self.cov_ = self.std_ = None
        # Accept single kernel (shared) or list (per-feature)
        if isinstance(kernel, list):
            self.kernels = kernel
            self._shared = False
        else:
            self.kernels = [kernel]
            self._shared = True

    def _build_prior_K(self, n_features):
        """
        Block-diagonal prior covariance.
        Each feature has its own kernel block — independent priors.
        Block-diagonal = features are a priori independent of each other.
        """
        ks = [self.kernels[0]]*n_features if self._shared else self.kernels
        n  = len(self.lags)
        K  = np.zeros((n_features*n, n_features*n))
        for i, k in enumerate(ks):
            K[i*n:(i+1)*n, i*n:(i+1)*n] = k(self.lags, self.lags)
        # Numerical jitter: prevents K from being exactly singular
        return K + 1e-6 * np.eye(K.shape[0])

    def fit(self, X, y):
        """
        Compute exact GP posterior in weight space.
        
        Steps:
        1. Build prior K from kernel(s)
        2. Posterior precision = K⁻¹ + XᵀX/σ²
        3. Posterior covariance Σ = precision⁻¹
        4. Posterior mean μ = Σ Xᵀy / σ²
        5. Marginal std = sqrt(diag(Σ))
        """
        n_times, n_weights = X.shape
        n_features = n_weights // len(self.lags)
        K     = self._build_prior_K(n_features)
        K_inv = np.linalg.inv(K)
        prec  = K_inv + X.T @ X / self.noise_var
        self.cov_  = np.linalg.inv(prec)
        self.mean_ = self.cov_ @ X.T @ y / self.noise_var
        self.std_  = np.sqrt(np.clip(np.diag(self.cov_), 0, None))
        return self

    def predict(self, X): return X @ self.mean_

    def score(self, X, y): return np.corrcoef(y, self.predict(X))[0, 1]

    def get_trf(self, name, slices):
        """
        Returns posterior mean and std for one feature.
        Plot mean ± 2*std for 95% credible interval.
        """
        sl = slices[name]
        return self.mean_[sl], self.std_[sl]

    def credible_interval(self, name, slices, z=2.0):
        m, s = self.get_trf(name, slices)
        return m - z*s, m + z*s

    def log_marginal_likelihood(self, X, y):
        """
        log p(y | X, θ) — objective for hyperparameter optimisation.
        
        Computed entirely in weight space via matrix determinant lemma:
          log|XKXᵀ + σ²I| = n·log(σ²) + log|K| + log|precision|
        
        And Woodbury for the quadratic form.
        Complexity: O(n_weights³ + n_times·n_weights²) — scales to real data.
        """
        if self.cov_ is None:
            self.fit(X, y)
        n_times, n_weights = X.shape
        n_features = n_weights // len(self.lags)
        K     = self._build_prior_K(n_features)
        K_inv = np.linalg.inv(K)
        prec  = K_inv + X.T @ X / self.noise_var
        _, ld_prec = np.linalg.slogdet(prec)
        _, ld_K    = np.linalg.slogdet(K)
        log_det_S  = n_times*np.log(self.noise_var) + ld_K + ld_prec
        Sxy  = self.cov_ @ (X.T @ y)
        quad = (y@y - (X.T@y)@Sxy/self.noise_var) / self.noise_var
        return -0.5 * (quad + log_det_S + n_times*np.log(2*np.pi))

    def optimise_hyperparams(self, X, y, verbose=True):
        """
        Learn kernel hyperparameters from data via empirical Bayes.
        
        Optimises in LOG SPACE to enforce positivity and handle
        the fact that hyperparams span orders of magnitude.
        
        Each feature gets its own output_scale and length_scale.
        noise_var is shared (one noise level per EEG channel).
        
        Uses L-BFGS-B — gradient-free, robust, standard choice.
        """
        n_times, n_weights = X.shape
        n_features = n_weights // len(self.lags)
        if self._shared:
            self.kernels = [self.kernels[0].copy() for _ in range(n_features)]
            self._shared = False

        def neg_lml(log_params):
            for i in range(n_features):
                self.kernels[i].output_scale = np.exp(log_params[2*i])
                self.kernels[i].length_scale = np.exp(log_params[2*i+1])
            self.noise_var = np.exp(log_params[-1])
            self.fit(X, y)
            return -self.log_marginal_likelihood(X, y)

        x0 = []
        for k in self.kernels:
            x0 += [np.log(k.output_scale), np.log(k.length_scale)]
        x0.append(np.log(self.noise_var))

        result = minimize(neg_lml, np.array(x0), method='L-BFGS-B',
                         options={'maxiter': 200, 'ftol': 1e-8})

        for i in range(n_features):
            self.kernels[i].output_scale = np.exp(result.x[2*i])
            self.kernels[i].length_scale = np.exp(result.x[2*i+1])
        self.noise_var = np.exp(result.x[-1])
        self.fit(X, y)

        if verbose:
            print("Optimised hyperparameters:")
            for i, k in enumerate(self.kernels):
                print(f"  Feature {i}: output_scale={k.output_scale:.4f}, "
                      f"length_scale={k.length_scale:.2f} samples "
                      f"({k.length_scale*1000/sr:.0f}ms at {sr}Hz)")
            print(f"  noise_var = {self.noise_var:.4f}  "
                  f"(true: {true_noise_var:.4f}, "
                  f"error: {abs(self.noise_var-true_noise_var)/true_noise_var*100:.1f}%)")
            print(f"  Converged : {result.success}")
        return self


# Fit GP-mTRF
kernel_list = [Matern52(1.0, 5.0), Matern52(0.5, 5.0)]
gp = GPmTRF(lags, kernel_list, noise_var=true_noise_var)
gp.optimise_hyperparams(X, eeg, verbose=True)

print(f"\nGP train score: {gp.score(X, eeg):.4f}")
print("GP-mTRF: ✓")

## Step 6: Results — TRF recovery and uncertainty

Now we compare:
1. **Ground truth** — the known TRF we're trying to recover
2. **Ridge** — point estimate, no uncertainty
3. **GP posterior mean** — should match truth at least as well as ridge
4. **GP credible interval** — the key advantage: where is the TRF uncertain?

A well-calibrated 95% credible interval should contain the true TRF
value at ~95% of lags. Small deviations from this are expected due to
empirical Bayes (hyperparams optimised on the same data used for inference).

In [ ]:
gp_mean_env,  gp_std_env  = gp.get_trf('envelope',  slices)
gp_mean_surp, gp_std_surp = gp.get_trf('surprisal', slices)
lo_env,  hi_env  = gp.credible_interval('envelope',  slices)
lo_surp, hi_surp = gp.credible_interval('surprisal', slices)

r_ridge_env  = np.corrcoef(true_trf_env,  ridge.get_trf('envelope',  slices))[0,1]
r_ridge_surp = np.corrcoef(true_trf_surp, ridge.get_trf('surprisal', slices))[0,1]
r_gp_env     = np.corrcoef(true_trf_env,  gp_mean_env)[0,1]
r_gp_surp    = np.corrcoef(true_trf_surp, gp_mean_surp)[0,1]

cov_env  = np.mean((true_trf_env  >= lo_env)  & (true_trf_env  <= hi_env))  * 100
cov_surp = np.mean((true_trf_surp >= lo_surp) & (true_trf_surp <= hi_surp)) * 100

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

def style(ax, title, xl='Lag (ms)', yl='Amplitude (a.u.)'):
    ax.set_title(title, fontsize=10, fontweight='500')
    ax.set_xlabel(xl, fontsize=9)
    ax.set_ylabel(yl, fontsize=9)
    ax.axhline(0, color='#CCCCCC', lw=0.8)
    ax.axvline(0, color='#CCCCCC', lw=0.8, ls='--')

# Panel 1: Ridge vs truth (envelope)
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(lag_ms, true_trf_env, color=COLORS['truth'], lw=2.2, label='ground truth')
ax1.plot(lag_ms, ridge.get_trf('envelope', slices), color=COLORS['ridge'],
         lw=1.8, ls='--', label=f'ridge (r={r_ridge_env:.4f})')
ax1.legend(fontsize=8, frameon=False)
style(ax1, 'Ridge — envelope')

# Panel 2: GP vs truth (envelope) with CI
ax2 = fig.add_subplot(gs[0, 1])
ax2.fill_between(lag_ms, lo_env, hi_env, alpha=0.18,
                 color=COLORS['ci'], label='95% credible interval')
ax2.plot(lag_ms, gp_mean_env, color=COLORS['gp'], lw=1.8,
         label=f'GP mean (r={r_gp_env:.4f})')
ax2.plot(lag_ms, true_trf_env, color=COLORS['truth'], lw=2.2, label='ground truth')
ax2.legend(fontsize=8, frameon=False)
style(ax2, f'GP-mTRF — envelope\nCI coverage={cov_env:.0f}%')

# Panel 3: overlay
ax3 = fig.add_subplot(gs[0, 2])
ax3.fill_between(lag_ms, lo_env, hi_env, alpha=0.12, color=COLORS['ci'])
ax3.plot(lag_ms, true_trf_env, color=COLORS['truth'], lw=2.2, label='ground truth')
ax3.plot(lag_ms, ridge.get_trf('envelope', slices), color=COLORS['ridge'],
         lw=1.6, ls='--', label='ridge')
ax3.plot(lag_ms, gp_mean_env, color=COLORS['gp'], lw=1.6, label='GP mean')
ax3.legend(fontsize=8, frameon=False)
style(ax3, 'Overlay — envelope')

# Panel 4: surprisal feature
ax4 = fig.add_subplot(gs[1, 0])
ax4.fill_between(lag_ms, lo_surp, hi_surp, alpha=0.18, color=COLORS['ci'])
ax4.plot(lag_ms, true_trf_surp, color=COLORS['truth'], lw=2.2, label='ground truth')
ax4.plot(lag_ms, ridge.get_trf('surprisal', slices), color=COLORS['ridge'],
         lw=1.6, ls='--', label=f'ridge (r={r_ridge_surp:.4f})')
ax4.plot(lag_ms, gp_mean_surp, color=COLORS['gp'], lw=1.6,
         label=f'GP mean (r={r_gp_surp:.4f})')
ax4.legend(fontsize=8, frameon=False)
style(ax4, f'Surprisal feature (weaker, +50ms)\nCI coverage={cov_surp:.0f}%')

# Panel 5: uncertainty per lag
ax5 = fig.add_subplot(gs[1, 1])
ax5.axvline(0, color='#CCCCCC', lw=0.8, ls='--')
ax5.plot(lag_ms, gp_std_env,  color=COLORS['gp'],   lw=1.8, label='envelope')
ax5.plot(lag_ms, gp_std_surp, color=COLORS['ridge'], lw=1.8, ls='--', label='surprisal')
ax5.legend(fontsize=8, frameon=False)
ax5.set_title('Posterior uncertainty per lag\n(ridge has no uncertainty)', fontsize=10)
ax5.set_xlabel('Lag (ms)')
ax5.set_ylabel('Posterior std (a.u.)')
# Surprisal should be more uncertain (weaker signal)

# Panel 6: summary metrics
ax6 = fig.add_subplot(gs[1, 2])
metrics = {
    'Ridge\nenvelope':   r_ridge_env,
    'GP\nenvelope':      r_gp_env,
    'Ridge\nsurprisal':  r_ridge_surp,
    'GP\nsurprisal':     r_gp_surp,
}
bars = ax6.bar(metrics.keys(), metrics.values(),
               color=[COLORS['ridge'], COLORS['gp'],
                      COLORS['ridge'], COLORS['gp']],
               width=0.6, edgecolor='white')
ax6.set_ylim(0.95, 1.002)
ax6.set_ylabel('TRF recovery (Pearson r)')
ax6.set_title('Recovery accuracy\n(both models vs ground truth)')
for bar, val in zip(bars, metrics.values()):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0003,
             f'{val:.4f}', ha='center', va='bottom', fontsize=8)

fig.suptitle('mixed-mTRF: Ridge vs GP — synthetic data validation',
             fontsize=12, fontweight='600', y=1.01)
plt.tight_layout()
plt.show()

print(f"\nEnvelope  — Ridge: r={r_ridge_env:.4f} | GP: r={r_gp_env:.4f} | CI coverage: {cov_env:.1f}%")
print(f"Surprisal — Ridge: r={r_ridge_surp:.4f} | GP: r={r_gp_surp:.4f} | CI coverage: {cov_surp:.1f}%")
print(f"True noise_var: {true_noise_var:.4f} | Learned: {gp.noise_var:.4f}")

## Step 7: Ridge = GP(WhiteNoise) — the proof

This is the key theoretical result.

**Ridge regression** solves $(X^\top X + \alpha I)^{-1} X^\top y$.

**GP with WhiteNoise kernel** sets $K = \frac{\sigma^2}{\alpha} I$ —
a diagonal prior where each lag is independent with variance $\sigma^2/\alpha$.

Substituting into the GP posterior mean formula:

$$\mu = \left(\frac{\alpha}{\sigma^2} I + \frac{X^\top X}{\sigma^2}\right)^{-1}
\frac{X^\top y}{\sigma^2}
= \left(X^\top X + \alpha I\right)^{-1} X^\top y$$

This is **exactly** ridge regression. The two are mathematically identical.

The difference: GP makes the prior **explicit** (via the kernel),
so you can replace it with something better — like Matérn 5/2.
Every mTRF paper ever published was implicitly assuming a WhiteNoise
prior. We just made that assumption visible, so we can change it.

In [ ]:
# Ridge and GP(WhiteNoise) must give identical weights
alpha     = 1.0
noise_var = 1.0

ridge_proof = RidgemTRF(lags, alpha=alpha)
ridge_proof.fit(X, eeg)

wn_kernel  = WhiteNoise(output_scale=noise_var / alpha)
gp_proof   = GPmTRF(lags, wn_kernel, noise_var=noise_var)
gp_proof.fit(X, eeg)

diff    = ridge_proof.coef_ - gp_proof.mean_
max_diff = np.max(np.abs(diff))

print(f"Max |ridge_coef − GP(WhiteNoise)_mean| = {max_diff:.2e}")
print(f"Machine epsilon (float64) ≈ 2.2e-16")
print(f"Difference is {'numerical zero ✓' if max_diff < 1e-6 else 'NOT zero — bug!'}")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.axhline(0, color='#CCCCCC', lw=0.8)
ax.axvline(0, color='#CCCCCC', lw=0.8, ls='--')
ax.plot(lag_ms, ridge_proof.get_trf('envelope', slices),
        color=COLORS['ridge'], lw=2, label='Ridge')
ax.plot(lag_ms, gp_proof.get_trf('envelope', slices)[0],
        color=COLORS['gp'], lw=1.5, ls='--', label='GP(WhiteNoise)')
ax.legend(fontsize=8, frameon=False)
ax.set_xlabel('Lag (ms)')
ax.set_ylabel('Amplitude (a.u.)')
ax.set_title('Ridge vs GP(WhiteNoise)\n— should be identical')

ax = axes[1]
ax.plot(np.arange(len(diff)), diff, color='#333', lw=1)
ax.axhline(0, color=COLORS['ridge'], lw=1.5, ls='--')
ax.set_title(f'Difference: Ridge − GP(WhiteNoise)\nmax|diff| = {max_diff:.2e}')
ax.set_xlabel('Weight index')
ax.set_ylabel('Difference')

plt.tight_layout()
plt.show()

## Summary

| | Ridge mTRF | GP-mTRF |
|---|---|---|
| **Prior** | Implicit WhiteNoise (independent lags) | Explicit smooth kernel (Matérn 5/2) |
| **Output** | Point estimate only | Posterior mean + uncertainty |
| **Hyperparams** | α (cross-validated) | output_scale, length_scale, noise_var (learned from data via LML) |
| **Scales to long recordings** | ✓ | ✓ (weight-space formulation) |
| **Uncertainty per lag** | ✗ | ✓ |
| **Special cases** | — | Ridge = GP(WhiteNoise) |

## Next steps

1. **Held-out hyperparameter tuning**: optimise hyperparams on first half of data,
   infer on full data — removes the empirical Bayes overconfidence
2. **Multi-channel support**: fit across all iEEG electrodes simultaneously
3. **Real iEEG data**: swap out `simulate.py` — all model code unchanged
4. **Group-level statistics**: mixed models / GAMMs across patients (SB-MEMA framework)
5. **Dynamic TRF extension**: model TRF latency as a function of visual predictability
   (analogous to Lalor 2024 but for audiovisual speech in intracranial data)